# Document Compressor Interface Reference

Developer-facing statements defined in `langchain_core.documents.compressor`.

# `BaseDocumentCompressor: BaseModel, ABC`

Abstract interface for post-processing retrieved documents using query context.

A concrete subclass must implement `compress_documents`. Users are encouraged to prefer a `RunnableLambda` instead of creating a subclass when practical.

## Required subclass hooks

### `compress_documents`

Compresses or otherwise post-processes retrieved documents for a query.

```python
compress_documents(
    self,
    documents: Sequence[Document], # Retrieved documents to process
    query: str, # Query providing the compression context
    callbacks: Callbacks | None = None, # Optional callbacks used during compression
) -> Sequence[Document] # Compressed or post-processed documents
```

## Methods

### `acompress_documents`

Asynchronously compresses retrieved documents for a query.

```python
async acompress_documents(
    self,
    documents: Sequence[Document], # Retrieved documents to process
    query: str, # Query providing the compression context
    callbacks: Callbacks | None = None, # Optional callbacks used during compression
) -> Sequence[Document] # Compressed or post-processed documents
```

The default implementation calls `compress_documents` through `run_in_executor`. Subclasses may override it to provide native asynchronous behaviour.

In [ ]:
#%pip install -U langchain-core#Install LangChain Core if it is not already installed

import re#Import regular expressions for extracting words
from collections.abc import Sequence#Import Sequence for the method type annotation
from langchain_core.callbacks import Callbacks#Import the callback type used by the compressor interface
from langchain_core.documents import Document#Import Document for storing text and metadata
from langchain_core.documents.compressor import BaseDocumentCompressor#Import the abstract compressor base class


class KeywordDocumentCompressor(BaseDocumentCompressor):#Create a compressor that filters documents using query keywords

    def compress_documents(self, documents: Sequence[Document], query: str, callbacks: Callbacks | None = None) -> Sequence[Document]:#Implement the required compression method
        query_words = set(re.findall(r"\b\w+\b", query.lower()))#Extract unique lowercase words from the query
        compressed_documents = []#Create an empty list for relevant documents

        for document in documents:#Process each retrieved document
            content_words = set(re.findall(r"\b\w+\b", document.page_content.lower()))#Extract unique lowercase words from the document
            matched_words = query_words.intersection(content_words)#Find words shared by the query and document

            if matched_words:#Keep the document only when at least one word matches
                updated_metadata = {**document.metadata, "matched_words": sorted(matched_words)}#Add matched words to the document metadata
                compressed_document = Document(page_content=document.page_content, metadata=updated_metadata)#Create the compressed document
                compressed_documents.append(compressed_document)#Add the relevant document to the result

        return compressed_documents#Return only the relevant documents


retrieved_documents = [#Create sample documents returned by a retriever
    Document(page_content="Users can reset their password from the account settings page.", metadata={"source": "account_guide.txt"}),#Create a password-related document
    Document(page_content="Refunds are processed within five to seven business days.", metadata={"source": "refund_policy.txt"}),#Create a refund-related document
    Document(page_content="Contact technical support when your account login fails.", metadata={"source": "support_guide.txt"}),#Create an account-support document
]#Finish the retrieved-document list

user_query = "How can I reset my account password?"#Define the user query
compressor = KeywordDocumentCompressor()#Create the custom compressor
compressed_documents = compressor.compress_documents(retrieved_documents, user_query)#Compress the retrieved documents

print(f"Documents before compression: {len(retrieved_documents)}")#Display the original number of documents
print(f"Documents after compression: {len(compressed_documents)}")#Display the remaining number of documents
print("=" * 70)#Display a separator

for document_number, document in enumerate(compressed_documents, start=1):#Process each relevant document
    print(f"Document {document_number}")#Display the document number
    print(f"Content: {document.page_content}")#Display the relevant document content
    print(f"Source: {document.metadata['source']}")#Display the document source
    print(f"Matched words: {document.metadata['matched_words']}")#Display the matching query words
    print("-" * 70)#Display a separator after each document